# Give your agent a document to search, plus a memory: Unstructured Transform MCP + mem0

Real-world documents (PDFs with columns, tables, scans) are hard to hand to a model directly.
[Unstructured](https://unstructured.io) solves this with **Unstructured Transform**, a hosted MCP server that turns
60+ file types into clean content. Since it is a remote
[MCP](https://modelcontextprotocol.io) server, you can plug it into the OpenAI **Responses API**
as a tool. Your agent gets document processing as a tool it can call on demand. For more information, check out the [docs](https://docs.unstructured.io/transform/overview). 10,000 free pages to start, then $0.015 a page.

Here is the idea, and it is the same way a coding agent works over a repository. Instead of chopping
the document into pieces and embedding them into a vector database, we:

1. Let the agent call the Transform tool to parse the PDF into **markdown**, and save that markdown
   to a local file.
2. Give the agent two simple tools over that file, `search_document` and `read_lines`, and let it
   search and read only the parts it needs. No embeddings, no vector database.
3. Use [mem0](https://mem0.ai) to remember **hints** about where common questions are answered, so
   the agent jumps to the right place faster next time. Feedback reinforces good hints.

The document stays on disk. Only the small slices the agent reads enter its context. mem0 stores
only hints, never the document. The demo document is the "Attention Is All You Need" paper.

## 1. Prerequisites

Three keys, read from environment variables (do not hard-code secrets):

- `UNSTRUCTURED_API_KEY` from https://unstructured.io/?modal=try-for-free (log in, then copy your API key)
- `MEM0_API_KEY` from https://app.mem0.ai
- `OPENAI_API_KEY` from https://platform.openai.com/api-keys

In [ ]:
%pip install --upgrade openai mem0ai requests

In [ ]:
import json
import os
import re
import time
from getpass import getpass

import requests
from mem0 import MemoryClient
from openai import OpenAI

for key in ("UNSTRUCTURED_API_KEY", "MEM0_API_KEY", "OPENAI_API_KEY"):
    if key not in os.environ:
        os.environ[key] = getpass(f"{key}: ")

client = OpenAI()       # reads OPENAI_API_KEY
mem = MemoryClient()    # reads MEM0_API_KEY

TRANSFORM_MCP_URL = "https://mcp.transform.unstructured.io/"
PDF_URL = "https://arxiv.org/pdf/1706.03762"  # Attention Is All You Need
DOC_PATH = "attention.md"
MODEL = "gpt-5"

## 2. Parse to markdown and save it locally

Parsing runs as an asynchronous job. We let the model drive the Transform tools, and we give it a
`wait_seconds` tool it can call between status checks. We describe the outcome we want rather than
naming tools, so the model reads what the server offers and picks for itself. Because `wait_seconds`
really sleeps, the agent can patiently wait for a slow job. We then fetch the markdown and save it to a
local file.

In [ ]:
def transform_mcp_tool():
    """Return a Responses API MCP tool block for the Unstructured Transform server."""
    return {
        "type": "mcp",
        "server_label": "unstructured_transform",
        "server_url": TRANSFORM_MCP_URL,
        "require_approval": "never",
        "headers": {"Authorization": f"Bearer {os.environ['UNSTRUCTURED_API_KEY']}"},
    }


def wait_seconds(seconds):
    """Wait before checking the job status again. The agent calls this so it can truly wait."""
    time.sleep(seconds)
    return f"Waited {seconds} seconds."


WAIT_TOOL = {
    "type": "function", "name": "wait_seconds",
    "description": "Wait the given number of seconds before checking the job status again.",
    "parameters": {"type": "object", "properties": {"seconds": {"type": "integer"}},
                   "required": ["seconds"], "additionalProperties": False},
}

MD_URL = re.compile(r"https://mcp\.transform\.unstructured\.io/output/\S+?\.md[^\"\\ ]*")


def parse_to_markdown(url):
    """Let the model drive Transform, with a real wait tool, and return the parsed markdown."""
    tools = [transform_mcp_tool(), WAIT_TOOL]
    instruction = (
        f"Parse the PDF at {url} into markdown using the Unstructured Transform tools. "
        "Submit the job with the URL, then poll its status, calling wait_seconds(10) "
        "between each check, until it is COMPLETED. Then fetch the results as markdown "
        "and report the download URL."
    )
    resp = client.responses.create(model=MODEL, tools=tools, input=instruction)
    seen = json.dumps(resp.model_dump())
    for _ in range(40):
        calls = [o for o in resp.output if o.type == "function_call"]
        if not calls:
            break
        outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = wait_seconds(**args) if call.name == "wait_seconds" else "unknown tool"
            outputs.append({"type": "function_call_output", "call_id": call.call_id, "output": result})
        resp = client.responses.create(model=MODEL, tools=tools,
                                       previous_response_id=resp.id, input=outputs)
        seen += "\n" + json.dumps(resp.model_dump())
    urls = MD_URL.findall(seen)
    if not urls:
        raise RuntimeError("Transform did not return a markdown URL; check the job status.")
    return requests.get(urls[0], timeout=120).text


markdown = parse_to_markdown(PDF_URL)
with open(DOC_PATH, "w", encoding="utf-8") as f:
    f.write(markdown)

print(f"Saved {len(markdown):,} characters to {DOC_PATH}")

## 3. Give the agent tools to search and read the file

Two small tools over the local markdown file. `search_document` returns matching lines with their
line numbers, and `read_lines` returns a range of lines. The agent decides what to search for and how
much to read, the same way a coding agent explores a repository. `run_agent` runs the short tool loop
and returns the final answer.

In [ ]:
def search_document(query):
    """Case-insensitive search of the local document. Returns matching lines with line numbers."""
    lines = open(DOC_PATH, encoding="utf-8").read().splitlines()
    hits = [f"L{i + 1}: {ln}" for i, ln in enumerate(lines) if query.lower() in ln.lower()]
    return "\n".join(hits[:25]) if hits else "No matches."


def read_lines(start, end):
    """Read a range of lines (1-indexed, inclusive) from the local document."""
    lines = open(DOC_PATH, encoding="utf-8").read().splitlines()
    start, end = max(1, start), min(len(lines), end)
    return "\n".join(f"L{i}: {lines[i - 1]}" for i in range(start, end + 1))


TOOL_FUNCS = {"search_document": search_document, "read_lines": read_lines}

TOOLS = [
    {"type": "function", "name": "search_document",
     "description": "Case-insensitive search of the document; returns matching lines with line numbers.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}},
                    "required": ["query"], "additionalProperties": False}},
    {"type": "function", "name": "read_lines",
     "description": "Read a range of lines (1-indexed, inclusive) from the document.",
     "parameters": {"type": "object",
                    "properties": {"start": {"type": "integer"}, "end": {"type": "integer"}},
                    "required": ["start", "end"], "additionalProperties": False}},
]

SYSTEM = (
    "You answer questions about a document you can only access through the search_document and "
    "read_lines tools. Search to locate the relevant part, read it, then answer from it only "
    "(no prior knowledge). End your answer with a line: 'Source: <the section heading you used>'."
)


def run_agent(question, hint_heading=None):
    system = SYSTEM
    if hint_heading:
        system += (f"\n\nMemory hint: a similar question was answered from the section titled "
                   f"'{hint_heading}'. Search there first.")
    resp = client.responses.create(
        model=MODEL, tools=TOOLS,
        input=[{"role": "system", "content": system}, {"role": "user", "content": question}],
    )
    for _ in range(6):
        calls = [o for o in resp.output if o.type == "function_call"]
        if not calls:
            return resp.output_text
        outputs = []
        for c in calls:
            args = json.loads(c.arguments)
            print(f"  tool call: {c.name}  args: {args}")
            result = TOOL_FUNCS[c.name](**args)
            preview = result.splitlines()[0] if result else ""
            print(f"    -> {preview[:90]}")
            outputs.append({"type": "function_call_output", "call_id": c.call_id, "output": result})
        resp = client.responses.create(
            model=MODEL, tools=TOOLS, previous_response_id=resp.id, input=outputs)
    return resp.output_text

## 4. Add memory: hints that make the agent faster

mem0 stores only hints: which section a kind of question was answered from. `answer()` looks for a
matching hint first (and passes it to the agent), answers, then saves a hint for next time.

In [ ]:
HINTS = "attention_hints"


def rows(res):
    return res.get("results", res) if isinstance(res, dict) else res


def get_hint(question):
    hits = rows(mem.search(question, version="v2", filters={"agent_id": HINTS}, limit=1))
    if hits:
        return hits[0]
    return None


def answer(question):
    hint = get_hint(question)
    hint_heading = (hint.get("metadata") or {}).get("heading") if hint else None
    reply = run_agent(question, hint_heading=hint_heading)

    heading = None
    match = re.search(r"Source:\s*(.+)", reply)
    if match:
        heading = match.group(1).strip()
    mem.add(messages=[{"role": "user", "content": question}],
            agent_id=HINTS, metadata={"heading": heading}, infer=False)

    print(f"[answered{' using a memory hint' if hint_heading else ''}; stored hint -> {heading}]")
    return reply

## 5. First question (nothing learned yet)

In [ ]:
print(answer("How does multi-head attention work?"))

## 6. A related question routes via memory

A differently worded question on the same topic finds the hint we just saved, which points the agent
at the right section. We then send positive feedback so mem0 favors that hint.

In [ ]:
question = "How many attention heads does the model use?"
used = get_hint(question)
print(answer(question))

if used:
    
    mem.feedback(memory_id=used["id"], feedback="POSITIVE")
    print("\nReinforced hint ->", (used.get("metadata") or {}).get("heading"))

## 7. Inspect what mem0 stored

Only hints. The document itself never went into mem0.

In [ ]:
for m in rows(mem.get_all(version="v2", filters={"agent_id": HINTS})):
    print(f"- {m.get('memory')}  (heading: {(m.get('metadata') or {}).get('heading')})")

## Conclusion

Unstructured Transform gives your agent document processing as a tool, so it can parse any document
on demand. Saving the result as markdown and handing the agent simple search and read tools lets it
navigate the document like a coding agent navigates a repository, pulling only what it needs into
context, with no vector database. mem0 adds a small memory of hints, so the agent gets faster on the
questions it sees often.

Next steps:
- Add a tool to read whole sections by heading, or return the nearest heading with each match.
- Bring in thresholds to prevent using unambiguous hints.
- Play around with what gets stored in memory, sometimes this works, sometimes you might need something else ;) 
- Swap in any file type Transform supports (PDF, DOCX, PPTX, images, and more). 

Resources:
- [Unstructured Transform](https://docs.unstructured.io/transform/overview)
- [mem0 docs](https://docs.mem0.ai) and [feedback](https://docs.mem0.ai/platform/features/feedback-mechanism)
